# Universal Chat Templating with `apply_chat_template`

This notebook demonstrates `thinkpack.apply_chat_template` — a universal wrapper that produces identical-looking prompts for any model, regardless of how that model handles reasoning blocks internally.

Different thinking models inject the opening reasoning tag in different ways:
- **Qwen3** (`prefixed=False`) — the model generates the open tag itself; the template does not inject it.
- **OLMo-3-Think** (`prefixed=True`) — the template appends `<think>` at the end of the generation prompt; the model generates the reasoning body itself.

Without a universal wrapper you would need different code for each model. With `apply_chat_template`, you write the same call once and it works correctly for all of them.

**What this notebook covers:**
1. `detect_model()` — auto-detecting template properties from a tokenizer
2. Basic inference prompts — same call, correct output for every model
3. Conversations with reasoning history — using a universal `reasoning` key
4. Thought steering — seeding the think block at inference time
5. Response prefix — seeding the model's response
6. Combining both prefixes
7. Batching with `apply_chat_templates`

In [10]:
import sys

# add the src directory to the path so thinkpack can be imported without installation
sys.path.insert(0, "../src")

from transformers import AutoTokenizer

from thinkpack import apply_chat_template, apply_chat_templates, detect_model

## 1. Detecting Model Properties

`detect_model()` inspects a tokenizer's chat template to determine:
- **prefixed** — `True` if the template injects the opening reasoning tag into the generation prompt, `False` if the model generates it itself
- **open_tag / close_tag** — the reasoning block delimiters used by this model

Detection is entirely behaviour-based — no template source scanning or hardcoded model names.

In [11]:
# load tokenizers for both models (cpu-only, no gpu needed)
tok_qwen = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-8B",
    trust_remote_code=True,
)
tok_olmo = AutoTokenizer.from_pretrained(
    "allenai/OLMo-3-7B-Think",
    trust_remote_code=True,
)

In [12]:
# detect template properties for each model — no manual configuration needed
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    info = detect_model(tok)
    print(f"=== {name} ===")
    print(f"  prefixed : {info.prefixed}")
    print(f"  open_tag : {info.open_tag!r}")
    print(f"  close_tag: {info.close_tag!r}")
    print()

=== Qwen3 ===
  prefixed : False
  open_tag : '<think>'
  close_tag: '</think>'

=== OLMo-3 ===
  prefixed : True
  open_tag : '<think>'
  close_tag: '</think>'



## 2. Basic Inference Prompts

The simplest use case: format a conversation ready for generation. The same call works for both models — `apply_chat_template` detects the template style and handles the generation primer correctly.

`apply_chat_template` takes a single conversation and returns a prompt string. Tokenization is left to the caller.

In [13]:
# a simple single-turn conversation
conversation = [
    {"role": "user", "content": "What is the capital of France?"},
]

# exact same call for both models
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
    )
    print(f"=== {name} ===")
    print(prompt)
    print()

=== Qwen3 ===
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
<think>


=== OLMo-3 ===
<|im_start|>system
You are OLMo, a helpful function-calling AI assistant built by Ai2. Your date cutoff is November 2024, and your model weights are available at https://huggingface.co/allenai. You do not currently have access to any functions. <functions></functions><|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
<think>



## 3. Multi-Turn Conversations with Reasoning History

When building conversation history that includes previous model reasoning (e.g. from a prior turn), use the `reasoning` key alongside `role` and `content` in assistant messages.

`apply_chat_template` translates the `reasoning` key by embedding it as literal `<think>...</think>` tags prepended to the message content. This works uniformly for all models — you never need to know which approach a given model's template uses.

In [14]:
# multi-turn conversation with a previous assistant turn that includes reasoning
conversation_with_history = [
    {
        "role": "user",
        "content": "What is 12 multiplied by 8?",
    },
    {
        "role": "assistant",
        # the universal 'reasoning' key — works for any model
        "reasoning": "12 × 8 = 96. I can verify: 12 × 8 = 12 × 4 × 2 = 48 × 2 = 96.",
        "content": "96",
    },
    {
        "role": "user",
        "content": "And what is that divided by 4?",
    },
]

# same call — reasoning is always embedded as inline <think>...</think> tags
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    prompt = apply_chat_template(
        conversation=conversation_with_history,
        tokenizer=tok,
    )
    print(f"=== {name} ===")
    print(prompt)
    print()

=== Qwen3 ===
<|im_start|>user
What is 12 multiplied by 8?<|im_end|>
<|im_start|>assistant
96<|im_end|>
<|im_start|>user
And what is that divided by 4?<|im_end|>
<|im_start|>assistant
<think>


=== OLMo-3 ===
<|im_start|>system
You are OLMo, a helpful function-calling AI assistant built by Ai2. Your date cutoff is November 2024, and your model weights are available at https://huggingface.co/allenai. You do not currently have access to any functions. <functions></functions><|im_end|>
<|im_start|>user
What is 12 multiplied by 8?<|im_end|>
<|im_start|>assistant
<think>
12 × 8 = 96. I can verify: 12 × 8 = 12 × 4 × 2 = 48 × 2 = 96.
</think>
96<|im_end|>
<|im_start|>user
And what is that divided by 4?<|im_end|>
<|im_start|>assistant
<think>



## 4. Thought Steering with `think_prefix`

At inference time, you may want to seed the model's reasoning — for example, to steer it towards a particular approach before it generates the rest of the think block itself.

`think_prefix` injects text at the start of the reasoning block. The model then continues generating from that point.

The same call works correctly for both template styles:
- **`prefixed=False` (Qwen3)** — the open tag is not yet in the prompt, so it is added before the prefix
- **`prefixed=True` (OLMo-3)** — the open tag is already at the end of the generation prompt, so only the body text is appended

In [15]:
# seed the model's thinking — the model will continue generating after this prefix
conversation = [
    {"role": "user", "content": "Is 97 a prime number?"},
]

for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
        think_prefix="Let me check by testing divisibility with primes up to √97 ≈ 9.8.",
    )
    print(f"=== {name} — tail of prompt ===")
    # show only the end of the prompt where the prefix was injected
    print(repr(prompt.split("assistant")[-1]))
    print()

=== Qwen3 — tail of prompt ===
'\n<think>\nLet me check by testing divisibility with primes up to √97 ≈ 9.8.'

=== OLMo-3 — tail of prompt ===
'\n<think>\nLet me check by testing divisibility with primes up to √97 ≈ 9.8.'



## 5. Response Prefix Injection

`response_prefix` seeds the model's response after the reasoning block closes. Setting it implicitly closes the think block first (equivalent to `close=True`), so the model is constrained to generate its response next.

This is useful for forcing a specific output format (e.g. JSON, a specific prefix like `"The answer is"`) or for constrained generation tasks.

In [16]:
# seed the response — think block closes automatically before the prefix
conversation = [
    {"role": "user", "content": "Is 97 a prime number?"},
]

for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
        response_prefix="The answer is: ",
    )
    print(f"=== {name} — tail of prompt ===")
    print(repr(prompt.split("assistant")[-1]))
    print()

=== Qwen3 — tail of prompt ===
'\nThe answer is: '

=== OLMo-3 — tail of prompt ===
'\n<think>\n</think>\nThe answer is: '



## 6. Combining Thought Steering and Response Prefix

`think_prefix` and `response_prefix` can be used together. The model reasons from the seeded prefix, then the think block is closed and the response starts from the seeded prefix.

This is particularly useful in constrained evaluation settings — you can direct the model's reasoning strategy while also enforcing a structured response format.

In [17]:
# combine think and response prefixes — model reasons from the seed, then responds in format
conversation = [
    {"role": "user", "content": "Is 97 a prime number?"},
]

for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
        think_prefix="Let me check divisibility with primes up to √97 ≈ 9.8.",
        response_prefix="The answer is: ",
    )
    print(f"=== {name} — tail of prompt ===")
    print(repr(prompt.split("assistant")[-1]))
    print()

=== Qwen3 — tail of prompt ===
'\n<think>\nLet me check divisibility with primes up to √97 ≈ 9.8.\n</think>\nThe answer is: '

=== OLMo-3 — tail of prompt ===
'\n<think>\nLet me check divisibility with primes up to √97 ≈ 9.8.\n</think>\nThe answer is: '



## 7. Batching with `apply_chat_templates`

`apply_chat_templates` (plural) accepts a list of conversations and processes them all in one call. `think_prefix` and `response_prefix` can be a single string (applied to every conversation) or a list with one entry per conversation — useful when you want different prefixes for different prompts.

In [18]:
# three questions processed in a single call with per-conversation think prefixes
conversations = [
    [{"role": "user", "content": "Is 97 a prime number?"}],
    [{"role": "user", "content": "What is 15% of 240?"}],
    [{"role": "user", "content": "What is the square root of 144?"}],
]

# different think seeds for each question
think_seeds = [
    "I'll test divisibility with small primes.",
    "I'll convert the percentage to a decimal first.",
    "I know that 12² = 144.",
]

# one call, three conversations, three different prefixes — works for any tokenizer
prompts = apply_chat_templates(
    conversations=conversations,
    tokenizer=tok_qwen,
    think_prefix=think_seeds,
)

for i, prompt in enumerate(prompts):
    print(f"=== Conversation {i + 1} — tail ===")
    print(repr(prompt.split("assistant")[-1]))
    print()

=== Conversation 1 — tail ===
"\n<think>\nI'll test divisibility with small primes."

=== Conversation 2 — tail ===
"\n<think>\nI'll convert the percentage to a decimal first."

=== Conversation 3 — tail ===
'\n<think>\nI know that 12² = 144.'



## Summary

| Feature | `Qwen3` (`prefixed=False`) | `OLMo-3` (`prefixed=True`) | Any other model |
|---|---|---|---|
| Basic prompt | Works | Works | Works |
| `reasoning` history key | → inline `<think>` tags in `content` | → inline `<think>` tags in `content` | → inline `<think>` tags in `content` |
| `think_prefix` | Adds `<think>\n` then prefix | Appends prefix after existing `<think>` | Adds `<think>\n` then prefix |
| `response_prefix` | Closes think block, adds prefix | Closes think block, adds prefix | Closes think block, adds prefix |
| Detection | Automatic | Automatic | Automatic |

The key point: **you write the same code once**. `detect_model()` handles the per-model differences internally, so your training and inference pipelines are identical regardless of which model family you are working with.